# NB05 — Interpretation & Dissemination
**Camino de Santiago — Contextual Drivers of Pilgrim Flow Mutations (2003–2025)**

Final analytical notebook. Synthesises quantitative findings (NB03–NB04b) against the behavioural framework (NB02b), produces stakeholder recommendations, and documents residual limitations.

| | |
|---|---|
| **Inputs** | `features_nb03.csv` · `NB04b_panel_results.csv` |
| **Outputs** | 4 figures · `NB05_recommendations.csv` |
| **Sections** | §1 Findings · §2 Triangulation · §3 Phases · §4 Limits · §5 Recommendations · §6 Synthesis |

## 0 · Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

ROOT     = Path('..').resolve()
PROCESSED = ROOT / 'data' / 'processed'
REPORTS  = ROOT / 'reports'
FIGURES  = ROOT / 'figures'
for p in [REPORTS, FIGURES]: p.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
plt.rcParams['figure.dpi'] = 120

matrix  = pd.read_csv(PROCESSED / 'features_nb03.csv', index_col='year').sort_index()
results = pd.read_csv(REPORTS / 'NB04b_panel_results.csv')

# Route colour palette (NB02 convention)
PALETTE = {
    'Camino Francés'        : '#2196F3',
    'Camino Portugués'      : '#E91E63',
    'Camino Portugués Costa': '#F44336',
    'Camino del Norte'      : '#4CAF50',
    'Camino Primitivo'      : '#9C27B0',
    'Camino Inglés'         : '#00BCD4',
    'Vía de la Plata'       : '#FF9800',
}
print('Setup OK')

---
## §1 · Five key findings

Each finding is stated in narrative form — statistical details are in NB04b; this section provides the interpretive layer.

### F1 — Climate effect confirmed at route level (Axis A)

A summer temperature anomaly of +1°C on a route corridor is associated with +23% pilgrim volume on that corridor (coef=+0.208, p=0.010, n=126, M1). Undetectable at n=21 — panel design was necessary to isolate this signal.

The positive sign applies to *Spanish* corridors: warmer, drier summers improve walking conditions. This is distinct from the Via Podiensis planning-lag mechanism (NB02 §4.1). Two climate mechanisms likely coexist at different spatial and temporal scales: one on *route selection* (France, lag=1), one on *volume* (Spain, contemporaneous).

**Operational implication:** seasonal temperature forecasts can serve as a leading indicator for expected pilgrim volume on specific corridors.

### F2 — COVID recovery was proportionally uniform

No DiD interaction was significant (p > 0.25 on all route-type interactions, M2). The rebound is strong (`post_covid` p=0.000) but non-differential — accessible, challenge and canonical routes recovered at the same proportional rate.

The structural shifts visible in the data (P.Costa growth, feminisation, diversification) pre-date COVID and continued through it. They are not COVID-induced reorientations — COVID was an interruption, not a pivot.

### F3 — French media interest is the dominant system-level driver

`trends_FR_lag0` is significant across all 6 core routes after route fixed effects (p=0.000, coef=+0.062, M3). Every unit increase in French Google Trends interest is associated with +6.4% system-wide volume.

**Mechanism (NB02b Profil B):** French media coverage — documentaries, testimonials, social media — primarily reaches the Secular Seeker profile (35–55, feminine, personal development motivation), which is the main engine of feminisation, secularisation and diversification observed since 2012.

### F4 — Portuguese media interest is route-specific, not systemic

`trends_PT_lag2` is non-significant in the 6-route panel (p=0.159) despite being the strongest NB02 predictor (r=0.938 with Camino Portugués share).

Resolution: trends_PT is a demand signal for the *Portuguese routes cluster* (Camino Portugués + P.Costa), not a system-wide driver. **Modelling rule:** include trends_PT in route-level models; exclude from system-level models.

### F5 — The 2018+ phase is media-driven, not an autonomous structural break

`phase_regime 3` (2018+) loses statistical significance when `trends_FR_lag0` is controlled for. The post-2018 acceleration is explained by the sustained rise of French media interest — not by an exogenous structural shock.

**Implication:** the three-phase periodisation remains a valid descriptive encoding, but the 2018 inflection is *caused* by the media trajectory.

In [ ]:
# ── §1 Summary chart — panel model R² ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

labels = [
    'M1\nClimate\n(Axis A)',
    'M2\nDiD COVID\n(route types)',
    'M3\nSystem\n(trends + events)',
]
r2_vals  = results['r2'].values
adj_vals = results['adj_r2'].values
x = np.arange(len(labels))
w = 0.3

ax.bar(x - w/2, r2_vals,  w, label='R²',     color='#1976D2', alpha=0.85)
ax.bar(x + w/2, adj_vals, w, label='adj-R²', color='#E53935', alpha=0.85)

# Annotate
for i, (r, a) in enumerate(zip(r2_vals, adj_vals)):
    ax.text(i - w/2, r + 0.005, f'{r:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(i + w/2, a + 0.005, f'{a:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('R²')
ax.set_ylim(0, 1.0)
ax.set_title('NB04b — Panel model performance (n=126, route × year)', fontsize=12)
ax.legend(fontsize=9)
ax.axhline(0.8, color='gray', lw=0.8, ls='--', alpha=0.5, label='R²=0.80 reference')

plt.tight_layout()
plt.savefig(FIGURES / 'NB05_panel_model_r2.png', dpi=150)
plt.show()

---
## §2 · Behavioural triangulation

Each finding is mapped against the three pilgrim profiles and three route clusters (NB02b) to assess behavioural coherence. Score: ✓✓ confirms · ✓ consistent · — neutral.

In [ ]:
# ── §2 Triangulation heatmap ─────────────────────────────────────────────────
# Rows = findings, Columns = behavioural profiles/clusters
# Score: +2 confirms, +1 consistent, 0 neutral, -1 inconsistent

profiles = ['Profil A\nTraditionnel', 'Profil B\nLaïc de quête', 'Profil C\nSportif-culturel',
            'Cluster\nCanonique', 'Cluster\nAccessibilité', 'Cluster\nDéfi']

findings = [
    'F1 — Climat +\n(corridor ES)',
    'F2 — COVID\nuniforme',
    'F3 — trends_FR\nsystémique',
    'F4 — trends_PT\nroute-spécifique',
    'F5 — 2018+\nmédiatique',
]

# Triangulation matrix (expert scoring from NB02b)
scores = np.array([
    # A    B    C   Canon  Access  Défi
    [ 0,   1,   2,    1,     1,     2],   # F1 climat: profil C marche par tous temps
    [ 1,   1,   1,    1,     1,     1],   # F2 COVID uniforme: tous profils
    [ 0,   2,   0,    0,     2,     0],   # F3 trends_FR: Profil B + routes accessibles
    [ 1,   2,   0,    1,     2,     0],   # F4 trends_PT: Profil B + cluster accessibilité
    [ 0,   2,   0,    0,     2,     1],   # F5 2018+ médiatique: Profil B dominant
])

fig, ax = plt.subplots(figsize=(11, 5))
cmap = plt.cm.RdYlGn
im = ax.imshow(scores, cmap=cmap, vmin=0, vmax=2, aspect='auto')

ax.set_xticks(range(len(profiles)))
ax.set_xticklabels(profiles, fontsize=9)
ax.set_yticks(range(len(findings)))
ax.set_yticklabels(findings, fontsize=9)

for i in range(len(findings)):
    for j in range(len(profiles)):
        label = {0:'—', 1:'✓', 2:'✓✓'}[scores[i,j]]
        ax.text(j, i, label, ha='center', va='center', fontsize=11,
                color='white' if scores[i,j] == 0 else 'black')

ax.set_title('Triangulation quantitatif × cadre comportemental NB02b\n'
             '✓✓ confirme · ✓ cohérent · — neutre', fontsize=11)
plt.colorbar(im, ax=ax, label='Cohérence comportementale')
plt.tight_layout()
plt.savefig(FIGURES / 'NB05_triangulation.png', dpi=150)
plt.show()

### Commentary

**Profil B (Secular Seeker) is the dominant engine** across F3, F4, F5. French and Portuguese media interest systematically points toward this profile and its preferred routes (P.Costa, Inglés, Portugués central). Feminisation, secularisation and geographic diversification are three statistical manifestations of the same underlying sociological shift.

**The climate signal (F1) aligns with Profil C (Athletic-Cultural).** Physical pilgrims are more sensitive to route conditions and more likely to plan based on weather information — consistent with the positive temperature-volume relationship in M1.

**COVID uniform recovery (F2) is profile-agnostic.** Pent-up demand affects all three profiles equally — no selective reorientation toward accessible or challenge routes.

---
## §3 · Structural phase narrative

The three-phase periodisation confirmed by NB02 PELT algorithm and NB04b can now be given a full behavioural interpretation.

In [ ]:
# ── §3 Phase timeline with PELT breaks and key events ────────────────────────
total = matrix['total_pilgrims']
diversity = matrix['diversity_core']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('Three structural phases of the Camino de Santiago (2004–2024)', fontsize=13)

# Phase backgrounds
for ax in [ax1, ax2]:
    ax.axvspan(2004, 2009, alpha=0.06, color='#607D8B', label='Phase 1 (2004–09)')
    ax.axvspan(2009, 2017, alpha=0.06, color='#1976D2', label='Phase 2 (2010–17)')
    ax.axvspan(2017, 2024, alpha=0.06, color='#388E3C', label='Phase 3 (2018+)')
    for yr, label in [(2009,'PELT break'), (2017,'PELT break'), (2020,'COVID'), (2021,'Holy Year')]:
        ax.axvline(yr, color='gray', lw=0.9, ls='--', alpha=0.6)

# Total pilgrims
ax1.fill_between(total.index, total/1000, alpha=0.25, color='#1976D2')
ax1.plot(total.index, total/1000, 'o-', color='#1976D2', lw=2, markersize=4)
ax1.set_ylabel('Pilgrims (thousands)')
ax1.set_title('Total pilgrims — three-regime trajectory')

# Annotations
ax1.annotate('Boom\npré-2010', xy=(2009, total.loc[2009]/1000),
             xytext=(2006, 260), fontsize=8, color='#607D8B',
             arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))
ax1.annotate('Recovery\n+44% vs 2019', xy=(2022, total.loc[2022]/1000),
             xytext=(2022.2, 350), fontsize=8, color='#388E3C')

# Diversity index
ax2.fill_between(diversity.index, diversity, alpha=0.25, color='#E53935')
ax2.plot(diversity.index, diversity, 'o-', color='#E53935', lw=2, markersize=4)
ax2.set_ylabel('diversity_core (1 − HHI)')
ax2.set_xlabel('Year')
ax2.set_title('System diversification — monotonic structural rise')
ax2.set_ylim(0, 1)

# Phase labels on ax2
for x, label in [(2006,'Phase 1\nStable'), (2013,'Phase 2\nDiversification'), (2020,'Phase 3\nPlateau')]:
    ax2.text(x, 0.08, label, ha='center', fontsize=8, color='#555555',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig(FIGURES / 'NB05_phase_narrative.png', dpi=150)
plt.show()

### Phase 1 — Stable regime (2004–2009)

Camino dominated by Profil A (Traditional): religious, male, European, Camino Francés. Francés share 77–85%, diversity index 0.28–0.39. The system is in stable cultural equilibrium around the religious Francés tradition. The 2009 PELT break *precedes* Holy Year 2010 — the acceleration was already underway.

### Phase 2 — Diversification boom (2010–2017)

*The Way* (2010, US release) triggers a structural break in % non-religious in 2012. Korean media interest begins predicting feminisation from 2011. Profil B enters at scale. All secondary routes grow: Norte ×2.5, Primitivo ×3, Inglés ×3 by 2017. Diversity index rises from 0.39 to 0.62. Francés monopoly structurally eroded but route still grows in absolute terms.

### Phase 3 — Recomposition plateau (2018+)

P.Costa emerges as a mass-market route (PELT inflection confirmed at 2018). Gender parity crosses 50% and never reverses. Diversity stabilises around 0.70 — P.Costa growth compensates route maturation. The acceleration is *caused by* French media interest (F5), not by an independent structural shock. COVID interrupts but does not reset the phase structure — 2022 recovery overshoots 2019 by 26%.

---
## §4 · Residual limitations

Consolidated from NB00 audit, NB02 methodology notes, and NB04b model constraints. Severity reflects impact on the robustness of results presented in §1.

In [ ]:
# ── §4 Limitations table ─────────────────────────────────────────────────────
limits = [
    ('L1', 'Compostela counting bias', '~40% undercount (Xunta IoT 2024)',
     'Trends valid; absolute volumes underestimated', 'Medium'),
    ('L2', 'n=126 panel ceiling', '6 routes × 21 years',
     'Max 5 features/model; no deep learning applicable', 'High'),
    ('L3', 'Trail SJ window', '2013–2024 only (n≈60 with trail features)',
     'Trail-dependent results indicative only', 'High'),
    ('L4', 'ERA5 proxy corridors', 'Primitivo/Inglés/Vía share Francés or Norte corridor',
     'Climate coefficients less precise for proxy routes', 'Medium'),
    ('L5', 'P.Costa sample', 'n=9 usable obs (2016–2024)',
     'No cross-validation, no causal claims', 'High'),
    ('L6', 'SJPDP data absent', 'Staged pilgrimage proxy only (pct_foot)',
     'Axis A sequential confound unquantified', 'Medium'),
    ('L7', 'Nationality data', '2004–2018 only (Oficina PDFs)',
     'Post-2018 nationality dynamics unobservable', 'Low'),
    ('L8', 'Porto aviation (OPO)', 'Data not collected (NB00)',
     'Low-cost hypothesis for P.Costa untestable', 'Medium'),
    ('L9', 'COVID decomposition', 'Structural vs conjunctural shift unresolved',
     '2025 data will provide first clean post-sanitary signal', 'Medium'),
]

lim_df = pd.DataFrame(limits, columns=['ID','Limitation','Detail','Impact','Severity'])

# Colour by severity
colors = {'High':'#FFCCCC','Medium':'#FFF3CC','Low':'#CCFFCC'}

fig, ax = plt.subplots(figsize=(15, 5))
ax.axis('off')
tbl = ax.table(
    cellText=lim_df[['ID','Limitation','Impact','Severity']].values,
    colLabels=['ID','Limitation','Impact on results','Severity'],
    cellLoc='left', loc='center',
    colWidths=[0.04, 0.22, 0.52, 0.10]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.6)

# Header style
for j in range(4):
    tbl[0,j].set_facecolor('#1F3864')
    tbl[0,j].set_text_props(color='white', fontweight='bold')

# Row colours
for i, (_, row) in enumerate(lim_df.iterrows(), start=1):
    for j in range(4):
        tbl[i,j].set_facecolor(colors[row['Severity']])

ax.set_title('Residual limitations — consolidated (NB00–NB04b)', fontsize=11,
             pad=12, loc='left')
plt.tight_layout()
plt.savefig(FIGURES / 'NB05_limitations.png', dpi=150, bbox_inches='tight')
plt.show()

---
## §5 · Stakeholder recommendations

Derived directly from quantitative findings and triangulated against the behavioural framework. Each recommendation is tagged with its source finding.

In [ ]:
# ── §5 Recommendations by stakeholder ───────────────────────────────────────
recs = [
    ('Xunta de Galice',
     'Infrastructure planning',
     'Use corridor summer temperature forecasts as a leading volume indicator per route. '
     'A +1°C anomaly → +23% expected volume on that corridor (F1).',
     'F1'),
    ('Xunta de Galice',
     'Capacity management',
     'COVID-type shocks produce uniform recovery across all route types — '
     'plan proportional capacity restoration, not differential reallocation toward '
     'accessible routes (F2).',
     'F2'),
    ('AFCC / SJPDP',
     'Communication strategy',
     'French media interest is the dominant system driver (F3). '
     'Monitoring Google Trends FR provides a 0-lag early warning of volume changes. '
     'Campaigns targeting Profil B (secular, feminine, 35–55) in French media '
     'have system-wide impact.',
     'F3, §2'),
    ('AFCC / SJPDP',
     'Route information',
     'The 2018+ growth is media-driven, not structurally autonomous (F5). '
     'Sustaining French media coverage is the highest-leverage action '
     'for maintaining pilgrim volume.',
     'F5'),
    ('Trail SJ by UTMB',
     'Positioning',
     'Trail culture growth is associated with deceleration of the challenge '
     'cluster (Norte/Primitivo/Vía de la Plata). Partial substitution confirmed. '
     'Trail SJ can position itself as the gateway for Profil C pilgrims, '
     'converting trail runners into Camino walkers rather than competing with them.',
     'NB04 T7'),
    ('Trail SJ by UTMB',
     'Data partnership',
     'Trail SJ finisher data (2013–2024, 11 editions) is the cleanest predictor '
     'of challenge cluster dynamics. Extending the historical series and '
     'nationality granularity would directly improve predictive models.',
     'L3'),
    ('Régions / Collectivités',
     'Séquençage FR→ES',
     'Le ratio pèlerins à pied est un proxy du pèlerinage séquentiel France→Espagne. '
     'Les données SJPDP (Saint-Jean-Pied-de-Port) amélioreraient significativement '
     'la modélisation de l\u2019Axe A et la planification des hébergements côté français.',
     'L6'),
    ('Xunta / Instituciones académicas',
     'Data collection',
     'Priority data gaps: Porto aviation routes (OPO) to test the low-cost P.Costa '
     'hypothesis (L8); post-2018 nationality data from Oficina (L7); '
     '2025 data to decompose COVID structural vs conjunctural shift (L9).',
     'L7, L8, L9'),
]

rec_df = pd.DataFrame(recs, columns=['Stakeholder','Domain','Recommendation','Source'])

# Print formatted
print('── Stakeholder recommendations ─────────────────────────────────────────')
current_stakeholder = ''
for _, row in rec_df.iterrows():
    if row['Stakeholder'] != current_stakeholder:
        print(f'\n▶ {row["Stakeholder"]}')
        current_stakeholder = row['Stakeholder']
    print(f'  [{row["Domain"]}]  (Source: {row["Source"]})')
    print(f'  {row["Recommendation"]}')

rec_df.to_csv(REPORTS / 'NB05_recommendations.csv', index=False)
print('\n✅  NB05_recommendations.csv saved')

---
## §6 · Project synthesis

### Axis status

| Axis | Status | Primary result |
|---|---|---|
| A — Climate | ✅ Confirmed (panel, p=0.010) | +1°C corridor → +23% volume |
| B — Media | ✅ Confirmed | trends_FR dominant system driver (p=0.000) |
| C — Diversification | ✅ Confirmed | 3-phase regime; 2018+ media-driven, not structural |
| D — Resilience | 🔄 Refined | COVID recovery uniform; Holy Year system-level amplifier |

### What the data cannot say

The Camino's structural mutations are **regime shifts**, not continuous contextual responses. Annual rate-of-change models fail for most targets — inflection points (2009, 2018, 2021) are discrete structural events whose triggers are identified (media trajectories, Holy Years) but whose full causal decomposition requires data not yet available (SJPDP, OPO, post-2018 nationalities).

### Research question — answer

> *'What contextual, non-demographic factors explain the structural mutations of the Camino de Santiago between 2003 and 2025?'*

**The dominant factor is French media interest** — a continuous, measurable demand signal that drives system-wide volume and explains the post-2018 acceleration without requiring an exogenous structural break.

**Climate** operates at the route-corridor level, confirming a secondary but real mechanism (+23% per +1°C anomaly).

**Trail culture** partially substitutes for challenge-route demand (Lasso R²=0.737 on growth_rate_challenge).

**Structural crises** (COVID, Holy Years) produce uniform system shocks — they amplify the existing trajectory rather than redirecting it.

In [ ]:
print('NB05 complete.')
print(f'Figures saved to: {FIGURES}')
print(f'Reports saved to: {REPORTS}')
print()
print('Files generated:')
for f in sorted(FIGURES.glob('NB05_*.png')):
    print(f'  {f.name}')
for f in sorted(REPORTS.glob('NB05_*.csv')):
    print(f'  {f.name}')